In [8]:
from google.colab import drive
import os
import pandas as pd
import os
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import numpy as np
import requests
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, losses, regularizers
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import time

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
ZIP_SRC  = "/content/drive/MyDrive/Deep Learning Project/GoogleStreetViewImagesNew.zip"
LOCAL_DIR = "/content/streetview_dataset"

if not os.path.exists(LOCAL_DIR):
    t0 = time.time()
    !cp "{ZIP_SRC}" /content/streetview_dataset.zip
    !unzip -q /content/streetview_dataset.zip -d {LOCAL_DIR}
    print(f"Staged in {time.time() - t0:.1f}s")

In [10]:
def dataloading(csv_path="/content/drive/MyDrive/Deep Learning Project/images_bra-2.csv",
                img_size=224,
                batch_size=32,
                preprocess_fn=None,
                ordinal=None):

    # 1. Load CSV
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=["income_group"]).reset_index(drop=True)

    # 2. Build file paths and group IDs
    df["full_path"] = df["file_path"].apply(
        lambda f: os.path.join("/content/streetview_dataset/", f)
    )
    df["loc_id"] = df["file_path"].apply(
        lambda f: os.path.basename(f).split("_h")[0]
    )

    # 3. Filter missing files before splitting
    before = len(df)
    df = df[df["full_path"].apply(os.path.exists)].reset_index(drop=True)
    print(f"Dropped {before - len(df)} missing files, {len(df)} remaining")

    # 4. Train / Val / Test split at location level
    locs = df["loc_id"].unique()
    np.random.seed(50)
    np.random.shuffle(locs)
    n_test = int(len(locs) * 0.15)
    n_val  = int(len(locs) * 0.15)
    test_locs  = locs[:n_test]
    val_locs   = locs[n_test:n_test + n_val]
    train_locs = locs[n_test + n_val:]

    train_df = df[df["loc_id"].isin(train_locs)]
    val_df   = df[df["loc_id"].isin(val_locs)]
    test_df  = df[df["loc_id"].isin(test_locs)]

    def to_ordinal(y, num_classes=3):
        out = np.zeros((len(y), num_classes - 1), dtype=np.float32)
        for i, label in enumerate(y):
            out[i, :label] = 1.0
        return out

    train_labels = train_df["income_group"].values.astype(np.int64)
    val_labels   = val_df["income_group"].values.astype(np.int64)
    test_labels  = test_df["income_group"].values.astype(np.int64)

    if ordinal:
        train_labels = to_ordinal(train_labels, num_classes=3)
        val_labels   = to_ordinal(val_labels,   num_classes=3)
        test_labels  = to_ordinal(test_labels,  num_classes=3)

    splits = {
        "train": (train_df["full_path"].values, train_labels),
        "val":   (val_df["full_path"].values,   val_labels),
        "test":  (test_df["full_path"].values,  test_labels),
    }

    for k, (p, l) in splits.items():
        print(f"{k}: {len(p)} images")

    # 5. Image loading function
    def load_image(path, label):
        raw = tf.io.read_file(path)
        img = tf.image.decode_jpeg(raw, channels=3)
        img = tf.image.resize(img, [img_size, img_size])
        img = tf.cast(img, tf.float32)
        if preprocess_fn:
            img = preprocess_fn(img)
        else:
            img = img / 255.0
        return img, label

    # 6. Build tf.data pipelines
    def make_dataset(file_paths, lbls, is_train=False):
        ds = tf.data.Dataset.from_tensor_slices((file_paths, lbls))
        ds = ds.cache()
        if is_train:
            ds = ds.shuffle(len(file_paths))
        ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
        return ds

    train_ds = make_dataset(*splits["train"], is_train=True)
    val_ds   = make_dataset(*splits["val"])
    test_ds  = make_dataset(*splits["test"])

    return train_ds, val_ds, test_ds


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image

MODEL_PATH = "/content/drive/MyDrive/Deep Learning Project/best_efficientnetv2s.keras"
CSV_PATH = "/content/drive/MyDrive/Deep Learning Project/images_bra-3.csv"
IMG_ROOT = "/content/streetview_dataset/"
IMG_SIZE = 384
N_CORRECT_PER_CLASS = 4          # how many correctly-classified images per class
OUTPUT_PNG = "/content/drive/MyDrive/Deep Learning Project/gradcam_grid.png"
CLASS_NAMES         = ["Low income", "Medium income", "High income"]
N_CLASSES           = len(CLASS_NAMES)

# Set to a list of absolute paths to bypass CSV sampling entirely.
IMAGE_PATHS = None

model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print(f"Loaded model: {model.name}")

# Locate the EfficientNetV2-S backbone nested inside the outer model.
base_model = None
for layer in model.layers:
    if isinstance(layer, tf.keras.Model) and "efficientnetv2" in layer.name.lower():
        base_model = layer
        break

if base_model is None:
    raise RuntimeError(
        "Could not locate an EfficientNetV2 sub-model. "
        "Inspect `model.layers` and update the search criterion."
    )
print(f"Backbone found: {base_model.name}")

LAST_CONV_LAYER_NAME = "top_activation"
last_conv_layer = base_model.get_layer(LAST_CONV_LAYER_NAME)
print(f"Grad-CAM target layer: {last_conv_layer.name}  "
      f"output shape: {last_conv_layer.output.shape}")

backbone_with_conv = tf.keras.Model(
    inputs=base_model.input,
    outputs=[last_conv_layer.output, base_model.output],
    name="backbone_with_conv_taps",
)

head_layers: list = []
_found = False
for layer in model.layers:
    if _found:
        head_layers.append(layer)
    elif layer is base_model:
        _found = True

if not head_layers:
    raise RuntimeError(
        "No head layers found after the backbone. "
        "Verify the outer model architecture."
    )
print(f"Head layers: {[l.name for l in head_layers]}")


def _run_head(backbone_out: tf.Tensor) -> tf.Tensor:
    """Pass backbone output through all head layers."""
    x = backbone_out
    for layer in head_layers:
        x = layer(x, training=False)
    return x


# GRAD-CAM
def compute_gradcam(
    img_array: np.ndarray,
    pred_index: int | None = None,
) -> tuple[np.ndarray, int, float, np.ndarray]:
    img_tensor = tf.cast(img_array, tf.float32)
    with tf.GradientTape() as tape:
        conv_act, backbone_out = backbone_with_conv(img_tensor, training=False)
        tape.watch(conv_act)
        preds = _run_head(backbone_out)
        if pred_index is None:
            pred_index = int(tf.argmax(preds[0]))
        class_score = preds[:, pred_index]

    grads = tape.gradient(class_score, conv_act)

    if grads is None:
        raise RuntimeError(
            "tape.gradient returned None.\n"
            "Possible causes:\n"
            "  • tape.watch(conv_act) was not called before _run_head\n"
            "  • head_layers is empty\n"
            "  • the backbone and head are not connected in the outer model"
        )

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))   # (C,)

    cam = conv_act[0] @ pooled_grads[..., tf.newaxis]       # (H', W', 1)
    cam = tf.squeeze(cam)                                   # (H', W')
    cam = tf.maximum(cam, 0.0)                              # ReLU
    cam = cam / (tf.reduce_max(cam) + 1e-10)               # normalise [0,1]

    return (
        cam.numpy(),
        int(pred_index),
        float(preds[0, pred_index]),
        preds[0].numpy(),
    )


def overlay_heatmap(
    orig_img: np.ndarray,
    heatmap: np.ndarray,
    alpha: float = 0.45,
    colormap=cm.jet,
) -> tuple[np.ndarray, np.ndarray]:
    heatmap_resized = (
        tf.image.resize(heatmap[..., np.newaxis], orig_img.shape[:2])
        .numpy()
        .squeeze()
    )
    heatmap_rgb = (colormap(heatmap_resized)[:, :, :3] * 255).astype(np.uint8)
    overlay = (orig_img * (1 - alpha) + heatmap_rgb * alpha).astype(np.uint8)
    return overlay, heatmap_resized


preprocess = tf.keras.applications.efficientnet_v2.preprocess_input

def load_and_preprocess(path: str) -> tuple[np.ndarray, np.ndarray]:
    orig = np.array(Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
    x = preprocess(orig.astype(np.float32))[np.newaxis, ...]
    return orig, x


if IMAGE_PATHS is None:
    print("\nLoading CSV and reconstructing test split...")
    df = pd.read_csv(CSV_PATH)
    df = df.dropna(subset=["income_group"]).reset_index(drop=True)
    if df["income_group"].dtype == object:
        # Try direct numeric cast first.
        try:
            df["income_group"] = df["income_group"].astype(int)
        except ValueError:
            # Fall back to name-based mapping.
            label_map = {name: i for i, name in enumerate(CLASS_NAMES)}
            df["income_group"] = df["income_group"].map(label_map)
            missing = df["income_group"].isna().sum()
            if missing:
                raise ValueError(
                    f"{missing} rows had income_group values not in CLASS_NAMES. "
                    f"Update CLASS_NAMES or the label_map."
                )
    df["income_group"] = df["income_group"].astype(int)

    df["full_path"] = df["file_path"].apply(lambda f: os.path.join(IMG_ROOT, f))
    df["loc_id"]    = df["file_path"].apply(
        lambda f: os.path.basename(f).split("_h")[0]
    )
    df = df[df["full_path"].apply(os.path.exists)].reset_index(drop=True)
    print(f"  Images found on disk: {len(df)}")
    np.random.seed(50)
    loc_ids = df["loc_id"].unique().copy()
    np.random.shuffle(loc_ids)
    n = len(loc_ids)
    test_ids = set(loc_ids[int(0.85 * n):])
    test_df = df[df["loc_id"].isin(test_ids)].reset_index(drop=True)
    print(f"  Test-set images: {len(test_df)}")

    print(
        f"\nScanning for ≥{N_CORRECT_PER_CLASS} correctly classified image(s) "
        f"per class..."
    )
    correct_per_class: dict[int, list[str]] = {i: [] for i in range(N_CLASSES)}

    for _, row in test_df.iterrows():
        # Early exit once every class bucket is full.
        if all(len(v) >= N_CORRECT_PER_CLASS for v in correct_per_class.values()):
            break
        true_label = int(row["income_group"])
        if len(correct_per_class[true_label]) >= N_CORRECT_PER_CLASS:
            continue        # this class is already full; skip inference
        try:
            _, x_input = load_and_preprocess(row["full_path"])
            _, pred_idx, _, _ = compute_gradcam(x_input)
            if pred_idx == true_label:
                correct_per_class[true_label].append(row["full_path"])
                print(f"  [{CLASS_NAMES[true_label]}] found #{len(correct_per_class[true_label])}: "
                      f"{os.path.basename(row['full_path'])}")
        except Exception as exc:
            print(f"  WARNING – skipping {row['full_path']}: {exc}")

    # Report and warn on any class with fewer samples than requested.
    all_ok = True
    for cls_idx, paths in correct_per_class.items():
        status = "✓" if len(paths) >= N_CORRECT_PER_CLASS else "✗"
        print(f"  {status} '{CLASS_NAMES[cls_idx]}': "
              f"{len(paths)}/{N_CORRECT_PER_CLASS} correctly classified samples")
        if len(paths) == 0:
            all_ok = False
    if not all_ok:
        raise RuntimeError(
            "Could not find any correctly classified sample for at least one class. "
            "The model may have very low accuracy on some classes, or the test set "
            "may be too small."
        )


    IMAGE_PATHS  = []
    TRUE_LABELS  = []
    for sample_idx in range(N_CORRECT_PER_CLASS):
        for cls_idx in range(N_CLASSES):
            if sample_idx < len(correct_per_class[cls_idx]):
                IMAGE_PATHS.append(correct_per_class[cls_idx][sample_idx])
                TRUE_LABELS.append(cls_idx)

else:
    TRUE_LABELS = [None] * len(IMAGE_PATHS)

print(f"\nFinal visualisation set: {len(IMAGE_PATHS)} images")


# PLOT
n_imgs = len(IMAGE_PATHS)
if n_imgs == 0:
    raise RuntimeError("IMAGE_PATHS is empty — nothing to plot.")

fig, axes = plt.subplots(n_imgs, 3, figsize=(12, 4 * n_imgs))
if n_imgs == 1:
    axes = axes[np.newaxis, :]

for i, (path, true_label) in enumerate(zip(IMAGE_PATHS, TRUE_LABELS)):
    orig, x_input = load_and_preprocess(path)

    heatmap, pred_idx, conf, all_preds = compute_gradcam(x_input)
    overlay, heatmap_resized = overlay_heatmap(orig, heatmap)

    pred_name = CLASS_NAMES[pred_idx]
    overlay_title = f"Pred: {pred_name} ({conf:.2f})"
    if true_label is not None:
        correct = pred_idx == int(true_label)
        overlay_title += f"\nTrue: {CLASS_NAMES[int(true_label)]}  {'✓' if correct else '✗'}"

    axes[i, 0].imshow(orig)
    axes[i, 0].set_title(f"Original\n{os.path.basename(path)}", fontsize=8)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(heatmap_resized, cmap="jet")
    axes[i, 1].set_title("Grad-CAM heatmap", fontsize=9)
    axes[i, 1].axis("off")

    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title(overlay_title, fontsize=9)
    axes[i, 2].axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=120, bbox_inches="tight")
print(f"\nSaved Grad-CAM grid → {OUTPUT_PNG}")
plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [17]:
n_rows = N_CORRECT_PER_CLASS
n_cols = N_CLASSES

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
if n_rows == 1:
    axes = axes[np.newaxis, :]
if n_cols == 1:
    axes = axes[:, np.newaxis]

for col, name in enumerate(CLASS_NAMES):
    axes[0, col].set_title(name, fontsize=12, fontweight="bold", pad=8)

for sample_idx in range(n_rows):
    for cls_idx in range(n_cols):
        ax = axes[sample_idx, cls_idx]
        flat_idx = sample_idx * N_CLASSES + cls_idx

        if flat_idx >= len(IMAGE_PATHS):
            ax.axis("off")
            continue

        path = IMAGE_PATHS[flat_idx]
        true_label = TRUE_LABELS[flat_idx]

        orig, x_input = load_and_preprocess(path)
        heatmap, pred_idx, conf, _ = compute_gradcam(x_input)
        overlay, _ = overlay_heatmap(orig, heatmap)

        ax.imshow(overlay)
        ax.set_xlabel(
            f"Pred: {CLASS_NAMES[pred_idx]} ({conf:.2f})",
            fontsize=8, labelpad=4
        )
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

plt.suptitle("Grad-CAM Overlays", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=120, bbox_inches="tight")
print(f"\nSaved Grad-CAM grid → {OUTPUT_PNG}")
plt.show()

Output hidden; open in https://colab.research.google.com to view.